**Linda Zier**

**ST 554**

**Final Project**

**Goal**

For this project we :

*   added our project and files to our github repo, committing often to show our progress.
*   wrote a Jupyter notebook that fits a machine learning model using pyspark’s MLlib module. In that same notebook we wrote code to read in a stream of data (data that we produced ourselves using a .py file that is also kept in the repo).
*   we used the model to do predictions on the stream and wrote those out to the console.


**Data**

The data is modified from the UCI machine learning repository. The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.


*   We used a chunk to build our model.
*   We then 'streamed data' to a folder that we monitored. As data came in we used our fitted model to predict on the incoming data.





# Train Models

We created a Jupyter notebook for the model fitting part and the streaming part below. We completed the following:

*   read the data into a standard pandas data frame using the pd.read_csv() function
*   converted this to a spark data frame
*   treated the Power_Zone_3 variable as our response variable and used the other variables as predictors

In [7]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA


spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


**Creating the Pipeline**

We fit an elastic net model using CV with the steps below.
The transformations below each used an MLlib function that we put into a pipeline.

*   We used an SQL transformer to cast the hour variable as a DoubleType.

*   We binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).

*   The month column was one-hot encoded.
*   We Ran a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. We did this by:
    - first by using a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator
    
    - then we had a PCA transformer for use in our pipeline.
    - we used two PCs in our transformation.


*   We renamed our response variable as label

*   We used VectorAssembler() to put our predictors into a features. The predictors are:

    – two fitted PCA features

    – binary Hour variable

    – Power_Zone_1

    – Power_Zone_2

    – Month indicator variables

This completes our pipeline of transformations!


In [14]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("transformations complete")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

transformations complete


In [16]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, pca_assembler, assembler])

print("pipeline complete")

pipeline complete
